# 模型推理 - 使用 QLoRA 微调后的 ChatGLM-6B

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# 模型ID或本地路径
model_name_or_path = 'THUDM/chatglm3-6b'

In [2]:
_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])

# 加载量化后模型(与微调的 revision 保持一致）
base_model = AutoModel.from_pretrained(model_name_or_path,
                                      quantization_config=q_config,
                                      device_map='auto',
                                      trust_remote_code=True,
                                      revision='b098244')

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

/data00/home/zhanguo/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
base_model.requires_grad_(False)
base_model.eval()

ChatGLMForConditionalGeneration(
  (transformer): ChatGLMModel(
    (embedding): Embedding(
      (word_embeddings): Embedding(65024, 4096)
    )
    (rotary_pos_emb): RotaryEmbedding()
    (encoder): GLMTransformer(
      (layers): ModuleList(
        (0-27): 28 x GLMBlock(
          (input_layernorm): RMSNorm()
          (self_attention): SelfAttention(
            (query_key_value): Linear4bit(in_features=4096, out_features=4608, bias=True)
            (core_attention): CoreAttention(
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (dense): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (post_attention_layernorm): RMSNorm()
          (mlp): MLP(
            (dense_h_to_4h): Linear4bit(in_features=4096, out_features=27392, bias=False)
            (dense_4h_to_h): Linear4bit(in_features=13696, out_features=4096, bias=False)
          )
        )
      )
      (final_layernorm): RMSNorm()
    )
    (output_la

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_chatglm.py: 0.00B [00:00, ?B/s]

## 使用原始 ChatGLM3-6B 模型

In [5]:
input_text = "解释下乾卦是什么？"

In [6]:
response, history = base_model.chat(tokenizer, query=input_text)

In [7]:
print(response)

乾卦是八卦之一，也是八宫图说、易经、易学等的重要概念之一。乾卦代表天、代表刚强、代表力量、代表阳、代表顺从、代表亨通。乾卦是由两个阴爻夹一个阳爻构成，象征着天父、君主、领导者的形象，也象征着阳刚之气、积极向上的精神状态。在八宫图说中，乾卦位于北方，与事业、努力、坚定、果敢等有关，与事业成功、发展、进步有关。在易经中，乾卦的卦辞为“元、亨、利、达”，象征着事物的顺利、成功、繁荣、发展。


#### 询问一个64卦相关问题（应该不在 ChatGLM3-6B 预训练数据中）

In [8]:
response, history = base_model.chat(tokenizer, query="周易中的讼卦是什么？", history=history)
print(response)

讼卦是八卦之一，也是八宫图说、易经、易学等的重要概念之一。讼卦代表诉讼、争端、诉讼、纷争、矛盾等。它是由两个阳爻夹一个阴爻构成，象征着阳刚之气的冲突和对立。在八宫图说中，讼卦位于西南，与法律、法律制度、法规、公正等有关，与法律诉讼、法律制度、法律问题等有关。在易经中，讼卦的卦辞为“决、辨、刚、义”，象征着通过辩论、分辨、刚正、正义等手段解决纷争和矛盾。


## 使用微调后的 ChatGLM3-6B

### 加载 QLoRA Adapter(Epoch=3, automade-dataset(fixed)) - 请根据训练时间戳修改 timestamp 

In [11]:
from peft import PeftModel, PeftConfig

epochs = 3
# timestamp = "20240118_164514"
timestamp = "20240225_222843"

peft_model_path = f"../models/{model_name_or_path}-epoch{epochs}"
print(peft_model_path)
config = PeftConfig.from_pretrained(peft_model_path)
qlora_model = PeftModel.from_pretrained(base_model, peft_model_path)
training_tag=f"ChatGLM3-6B(Epoch=3, automade-dataset(fixed))"

../models/THUDM/chatglm3-6b-epoch3


In [12]:
def compare_chatglm_results(query, base_model, qlora_model, training_tag):
    base_response, base_history = base_model.chat(tokenizer, query)

    inputs = tokenizer(query, return_tensors="pt").to(0)
    ft_out = qlora_model.generate(**inputs, max_new_tokens=512)
    ft_response = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    
    print(f"问题：{query}\n\n原始输出：\n{base_response}\n\n\n微调后（{training_tag}）：\n{ft_response}")
    return base_response, ft_response

### 微调前后效果对比

In [13]:
base_response, ft_response = compare_chatglm_results("解释下乾卦是什么？", base_model, qlora_model, training_tag)

问题：解释下乾卦是什么？

原始输出：
乾卦是周易中的一卦，代表天，象征刚健强劲，充满活力。它由两个乾卦叠加而成，原始状态是震卦下，巽卦上。乾卦的卦象是干燥的云彩，象征着干燥、强劲、果敢的特点。

在周易中，乾卦代表天，反映了刚健强劲的特性。同时，它也象征着果敢、坚毅和充满活力的态度。在咨询卦象时，乾卦表示当前时机适合冒险，可以冒险可以去远行。

在命理学中，乾卦具有积极刚健的形象，适用于年龄较大的人，可以适用于创业、求名、婚恋、移徙等事务。同时，乾卦也具有冒险精神，适合在较早的阶段进行尝试和冒险。

乾卦所蕴含的哲学观念是：顺应天道，刚健强劲，果敢坚毅，充满活力。在实际生活中，我们需要学会在遵循天道的基础上，保持刚健强劲的精神，勇敢面对挑战，坚毅不变，以此在各个方面取得成功。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset(fixed))）：
[gMASK]sop 解释下乾卦是什么？ 乾卦是周易中的一卦，代表天，象征刚健强劲，充满力量。它由两个乾卦叠加而成，其中上面一个乾卦代表天，下面一个乾卦代表干燥，两者结合代表干燥之天，故称为乾卦。

乾卦的核心象征是刚健强劲，它代表一种强烈的生命力、决断力和执行力。它预示着干燥正规，干燥缺乏雨水，但仍然可以依赖干燥的云层来滋润，从而使种子发芽。

在周易中，乾卦代表一种刚健强劲的力量，预示着干燥正规，干燥缺乏雨水，但仍然可以依赖干燥的云层来滋润，从而使种子发芽。


In [14]:
base_response, ft_response = compare_chatglm_results("周易中的讼卦是什么", base_model, qlora_model, training_tag)

问题：周易中的讼卦是什么

原始输出：
讼卦是周易卦象中的一枚卦，由上卦坤（地）和下卦乾（天）组成，象征着天地的相争，具有 Schema 卦象。在卜筮讼卦时，利于见利如仓，有利于客（客卦）。

讼卦的核心哲学是：天（乾）与地（坤）相争，寒气射天，客（坤）养天， both benefit well。

讼卦的传统解卦是：初入大利，有贵人相助，虽有利，但小利珍取。

讼卦的现代解卦是：充满活力，追求创新，具有开拓精神，注重人际关系。

讼卦的运势解卦是：利他人，不宜争斗，和平相处，和平决策。

讼卦的生活应用时间是：有利于解决争端，但不利于过于刚毅。

讼卦的运势应用时间是：在竞争中，的优势在于和平、合作、温和态度，避免争斗。

讼卦的的事业应用时间是：有利于竞争，但不适合过于刚毅，合作较好，注重效率。

讼卦的婚姻应用时间是：有利于订婚、嫁娶，但不适合过于刚毅的夫妻，注重相互理解和包容。

讼卦的交友应用时间是：有利于与贵人交往，但不适合与敌对势力交往，避免争端。

讼卦的智慧解卦是：柔能克刚，和平详和，利端在于不敢争。

讼卦的解卦是：天地的相争，寒气射天，客（坤）养天， both benefit well。

讼卦的决卦是：和平、合作、温和态度，避免争斗。

讼卦的FAQ是：

1. 讼卦为什么利？
   - 利在于仓，有利于客。

2. 讼卦的卦象意味着什么？
   - 利于见利如仓，有利于客。

3. 讼卦在竞争中如何？
   - 在竞争中，优势在于和平、合作、温和态度，避免争斗。

4. 讼卦在生活中的应用？
   - 有利于解决争端，但不利于过于刚毅。

5. 讼卦在事业上的应用？
   - 有利于竞争，但不适合过于刚毅，合作较好，注重效率。

6. 讼卦在婚姻上的应用？
   - 有利于订婚、嫁娶，但不适合过于刚毅的夫妻，注重相互理解和包容。

7. 讼卦在交友上的应用？
   - 有利于与贵人交往，但不适合与敌对势力交往，避免争端。

8. 讼卦为什么有利于和平、合作、温和态度？
   - 柔能克刚，和平详和，利端在于不敢争。

9. 讼卦的解卦如何？
   - 天地的相争，寒气射天，客（坤）养天， both benefit well。

10. 讼卦的决卦如何？
    - 和平、合作、温和态度，避免争斗。


微调后（ChatGLM3-6B(Epo

In [15]:
base_response, ft_response = compare_chatglm_results("师卦是什么？", base_model, qlora_model, training_tag)

问题：师卦是什么？

原始输出：
师卦是一个由坎卦（水）上承坤卦（地）组成的卦象，代表军队和指挥军情的卦象。根据《象辞》，这一卦象被解释为“地中有水”，象征着像大地一样包容和养育众人。根据《断易天机》，只有德高望重的长者来统率军队，才能获得吉祥无咎。


据北宋易学家邵雍解，得师卦者将面临困难重重，忧心劳众，宜包容别人，艰苦努力，摒除一切困难。台湾国学大儒傅佩荣解则提到，对于时运、财运、家宅和身体等方面会有相应影响。


传统解卦认为，师卦具有养兵聚众、出师攻伐之象，彼此有伤，难得安宁的大象。在运势方面，预示着困难重重，需要以正规行事，谨小慎微，严于律已。在事业、经商、求名、婚恋和决策等方面，都需要保持冷静、谨慎，注意避免敌人和困难带来的不利影响，必能成功。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset(fixed))）：
[gMASK]sop 师卦是什么？ 师卦是一个由坎卦（水）上承坤卦（地）组成的卦象，代表军队和指挥军情的卦象。根据《象辞》，这一卦象被解释为“地中有水”，象征着像大地一样包容和养育众人。根据《断易天机》，只有德高望重的长者来统率军队，才能获得吉祥无咎。


据北宋易学家邵雍解，得师卦者将面临困难重重，忧心劳众，宜包容别人，艰苦努力，摒除一切困难。台湾国学大儒傅佩荣解则提到，对于时运、财运、家宅和身体等方面会有相应影响。


传统解卦认为，师卦具有养兵聚众、出师攻伐之象，彼此有伤，难得安宁的大象。在运势方面，预示着困难重重，需要以正规行事，谨小慎微，严于律已。在事业、经商、求名、婚恋和决策等方面，都需要保持冷静、谨慎，注意避免敌人和困难带来的不利影响，必能成功。


## 其他模型（错误数据或训练参数）

#### 加载 QLoRA Adapter(Epoch=3, automade-dataset)

In [16]:
from peft import PeftModel, PeftConfig

epochs = 3
peft_model_path = f"../models/{model_name_or_path}-epoch{epochs}"

config = PeftConfig.from_pretrained(peft_model_path)
qlora_model_e3 = PeftModel.from_pretrained(base_model, peft_model_path)
training_tag = f"ChatGLM3-6B(Epoch=3, automade-dataset)"

In [17]:
base_response, ft_response = compare_chatglm_results("解释下乾卦是什么？", base_model, qlora_model_e3, training_tag)

问题：解释下乾卦是什么？

原始输出：
乾卦是周易中的一卦，代表天，象征刚健强劲的特性。它由六个阳爻组成，相应于周易中的乾（天）卦。在卜卦时，阳爻表示积极刚健的特性，阴爻表示柔顺乃至静的特性。

乾卦的卦辞是：“元、亨、利、贞”，意味着充满了力量、刚健和正义，是吉利之卦。

乾卦的象征意义是：天行健，君子以健忘之志；天行健，君子以利行。这一卦教导人们要积极刚健，行走在吉利之上。

在周易中，乾卦所传达的哲学观念是：世界是充满力量、刚健和正义的，人们应该积极刚健，行走在吉利之上。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset)）：
[gMASK]sop 解释下乾卦是什么？ 乾卦是周易中的一卦，代表天，象征刚健强劲，充满力量。它由两个乾卦叠加而成，其中上面一个乾卦代表天，下面一个乾卦代表干燥，两者结合代表干燥之天，故称为乾卦。

乾卦的核心象征是刚健强劲，它代表一种强烈的生命力、决断力和执行力。它预示着干燥正规，干燥缺乏雨水，但仍然可以依赖干燥的云层来滋润，从而使种子发芽。

在周易中，乾卦代表一种刚健强劲的力量，预示着干燥正规，干燥缺乏雨水，但仍然可以依赖干燥的云层来滋润，从而使种子发芽。


In [18]:
base_response, ft_response = compare_chatglm_results("地水师卦是什么？", base_model, qlora_model_e3, training_tag)

问题：地水师卦是什么？

原始输出：
地水师卦是易经中的卦象，由上卦坤（地），下卦坎（水）组成，预示着可成功并及时得到他人帮助。地水象征着地面上的水，代表着润泽、滋养、宁静和安详的特点。在占卜中，这个卦象提醒人们要谦卑谨慎，遵循湿润柔和的品质，可以获得吉祥。

地水师卦具有深远的含义，提示人们在面对困境时，要谦卑谨慎，保持冷静，遵循湿润柔和的品质，可以获得吉祥。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset)）：
[gMASK]sop 地水师卦是什么？ 地水师卦是易经卦象中的一枚卦，由上卦坤（地）和下卦坎（水）组成，预示着吉利的变化。具体来说，根据《象辞》，这一卦意味着“地中有水”，象征着地面上的河流和湖泊。根据《断易天机》，地水师卦预示着吉祥如意，具有刚柔之志，能化敌为友。在塔罗牌中，地水师卦表示情感稳定，生活幸福。

地水师卦的哲学含义是：地面上有河流和湖泊，水滋润着土地，使土地更加肥沃。这表明，在人生过程中，需要注重积累德行和智慧，以柔克刚，刚柔并济，以实现吉祥如意。

地水师卦的 predictive意义是：预示着吉利如意，刚柔之志，能化敌为友。在事业、经商、求名、婚恋等方面都有积极的影响。

在塔罗牌中，地水师卦表示情感稳定，生活幸福。在解牌时，需要结合牌面的具体内容，综合考虑各种因素，以获得准确的解读。


In [19]:
base_response, ft_response = compare_chatglm_results("周易中的讼卦是什么", base_model, qlora_model_e3, training_tag)

问题：周易中的讼卦是什么

原始输出：
讼卦是周易卦象中的一枚卦，由上卦坤（地）和下卦乾（天）组成，象征着天地的相争，具有刚柔相济的特性。在卜卦时，讼卦预示着利见凶险，必须谨慎避让对方，以利为安为上策。讼卦的哲学深度体现在天地的相争中，刚柔相济的特性，以及利害关系的变化上。

讼卦的核心哲学是：

- 利见凶险
- 必须谨慎
- 利害关系
- 变化 matters

讼卦所蕴含的智慧适用于生活中的各种挑战和冲突。在解卦时，卜者需要以冷静的态度，明辨是非，刚柔相济，以利为安为上策。在应对讼卦时，决策者应更加谨慎，以避免冲突的升级，以实现和平解决问题。

讼卦的启示是：

- 利见凶险
- 必须谨慎
- 利害关系
- 变化 matters

在实际应用中，讼卦提示我们在面临冲突和争议时，需要保持冷静和谨慎，以避免冲突升级，以利为安为上策。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset)）：
[gMASK]sop 周易中的讼卦是什么卦象

 讼卦是周易卦象中的一卦，由下卦坎（水）上卦乾（天）组成，代表的天水相交，象征水在阳光下闪闪发光。讼卦具有刚健之志，但容易犯刚强之过，因此需要谦逊谦逊，注意分寸。

讼卦的核心哲学是：

- 卦象：上卦为乾，下卦为坎
- 象义：乾为天，为刚；坎为水，为柔
- 卦情：刚柔相济，刚强不足
- 调护全局：刚强不足，需要柔顺，需要分寸

讼卦的运势：

- 事业：初有成功，但 soon to be overcome，需谦逊谨慎
- 财运：刚有收获，但 soon to be overcome，需节俭
- 感情：刚有收获，但 soon to be overcome，需真诚
- 事业指数：0.8
- 事业趋势：初有成功，但soon to be overcome
- 事业策略：谦逊、分寸

讼卦的生意指数：

- 开始时顺利，但不久将遇到阻碍
- 开始时业务繁荣，随后将遇到困难
- 需要耐心等待
- 生意指数：0.6

讼卦的运势指数：

- 开始时顺利，但不久将遇到困难
- 开始时顺利，但不久将遇到困难
- 需要耐心等待
- 运势指数：0.6

讼卦的解析：

- 初始时顺利，但 soon to be overcome，因此需谦逊、分寸
- 开始时业务繁荣，但随后将遇到困难
- 需要耐心等待
- 初始顺利，但

#### 加载 QLoRA Adapter(Epoch=50, Overfit, handmade-dataset)

In [18]:
from peft import PeftModel, PeftConfig

epochs = 50
peft_model_path = f"models/{model_name_or_path}-epoch{epochs}"

config = PeftConfig.from_pretrained(peft_model_path)
qlora_model_e50_handmade = PeftModel.from_pretrained(base_model, peft_model_path)
training_tag = f"ChatGLM3-6B(Epoch=50, handmade-dataset)"

In [19]:
base_response, ft_response = compare_chatglm_results("解释下乾卦是什么？", base_model, qlora_model_e50_handmade, training_tag)

问题：解释下乾卦是什么？

原始输出：
乾卦是《易经》中的第一卦，卦象由六个阳爻（即阳性的木）排列而成，它们从下至上排列，看起来像是天上的云彩，因此得名“乾”。乾卦象征着刚强、坚毅和独立，以及与天地之间的联系。

乾卦的卦辞是：“乾为天，为父，为君，亲阳人。亲，先祖，家远的代际，宜先祖；国，先王，宜治国；夫，配偶，宜婚配；父，尊亲，宜尊父；君，朋友，宜友道。”卦辞描述了乾卦的内涵，它既象征着天，也象征着人。作为天，它与“先祖”和“先王”有关，表示宇宙的起点和万物的基础。作为人，它与“父”和“君”有关，表示人类的社会性和文化传承。

在《易经》中，乾卦是“八卦”之一，它既象征着阳，也象征着“刚”和“强”。乾卦的卦象既象征着“天”，也象征着“人”，它是“八卦”中最为重要的卦之一，表示宇宙的运行和万物的生成。


微调后（ChatGLM3-6B(Epoch=50, handmade-dataset)）：
[gMASK]sop 解释下乾卦是什么？乾卦是《易经》中的第一卦，也是六十四卦中的首卦，它的卦象是由两个阳爻夹一个阴爻构成，象征着天，也象征着刚强和坚毅。乾卦的卦辞是“元，亨，利，贞”，意味着“ beginning, success, profit, integrity”。它表示万事万物的本源和原理，是成功的基础。乾卦 also means "the heaven", and it is a symbol of the universe, with six lines of yin and one line of yang. It represents the power and strength of the world, and is a symbol of integrity.


In [20]:
base_response, ft_response = compare_chatglm_results("地水师卦", base_model, qlora_model_e50_handmade, training_tag)

问题：地水师卦

原始输出：
师卦（师的卦象为乾为天，地为地）主动向被动，主动而被动，地震师，吉。象征师兵，兵者国之大事，故放师卦于和创新，守师卦则不进。

师卦是由天（乾）和地（坤）组成的卦象，乾为天，坤为地。师卦是《易经》中第八卦，属于坤卦（ earth）系列卦象，坤卦下卦为坤，上卦为乾，乾为天，坤为地。

师卦象征着师兵，主动向被动，主动而被动，地震师，吉。象征师兵，兵者国之大事，故放师卦于和创新，守师卦则不进。

师卦最佳状态为师的（师卦上卦为乾，下卦为坤），乾为天，坤为地，师卦象征师兵，主动向被动，主动而被动，地震师，吉。象征师兵，兵者国之大事，故放师卦于和创新，守师卦则不进。

在占卦中，师卦卦象为吉凶相生，以阳刚为上，以顺为正，symbolism of the earth and sky，象征天地。师卦：师 师 师。三个师字，下坤上乾。

师卦卦象分析：师卦是乾卦下坤卦上，乾为天，坤为地，象征天和地。师卦象征师兵，主动向被动，主动而被动，地震师，吉。

师卦卦象占卜：占师卦，得师卦，吉。

师卦卦象应用：师卦象征师兵，主动向被动，主动而被动，象征天和地。适用于天和地的事物，占师卦 Creature and earth。


微调后（ChatGLM3-6B(Epoch=50, handmade-dataset)）：
[gMASK]sop 地水师卦 师卦原文：师。贞，丈人吉，无咎。象曰：地中有水，师。君子以容民畜众。白话文解释：师卦象征军队指挥，无灾祸。《象辞》说：下卦为坎（水），上卦为坤（地），如大地容纳江河，君子应容纳众人。《断易天机》解：师卦坤上坎下，象征军众，需德高长者统率以吉无咎。北宋易学家邵雍解：忧劳动众，公正无私排难。得卦者应包容他人，努力排除困难。台湾国学大儒傅佩荣解：时运包容他人，财运有财需珍惜，家宅旧亲联姻吉，身体腹胀调气。传统解卦：异卦（下坎上坤），“师”指军队。坎为水险，坤为地顺，寓兵于农，用兵应顺势，故化凶为吉。


In [21]:
base_response, ft_response = compare_chatglm_results("天水讼卦", base_model, qlora_model_e50_handmade, training_tag)

问题：天水讼卦

原始输出：
讼卦（卦名：讼）是《易经》中的第八卦，属于雷（卦头）坎（卦体）离（卦象）两种卦象的组合。讼卦象征诉讼、争议、争端等诉讼方面的现象。卦形为雷上坎下，雷象征刚毅，坎象征险，离象征烈。卦象下卦为坎，上卦为离，险险相间，刚毅相结合。

讼卦的卦辞：“讼，吉凶交战，得利藏损。”卦象下坎上离，坎险离光，光争下，险林光，刚毅争利，光损藏吉。

蔽卦（卦名：蔽）是《易经》中的第六卦，属于离（卦头）坤（卦体）震（卦象）两种卦象的组合。蔽卦象征屏蔽、遮蔽，阻止邪恶势力，弘扬正道。

蔽卦的卦辞：“蔽，利ulo，吉凶凶险皆屏蔽。”卦象下离上坤，离火遮蔽，坤地顺从，刚毅相合，和顺利害，吉祥安全。

 src="http://www.qk6.com/article/html/277776.html" target="_blank">http://www.qk6.com/article/html/277776.html


微调后（ChatGLM3-6B(Epoch=50, handmade-dataset)）：
[gMASK]sop 天水讼卦 讼卦原文：讼。有孚，窒惕，中吉，终凶。利见大人，不利涉大川。象曰：天与水违行，讼。君子以做事谋始。白话文解释：讼卦象征虽有利可图但需警惕。事情初吉后凶，利于见贵人，不宜涉水。《象辞》说：上卦为乾（天），下卦为坎（水），天水相隔，事理不合，君子需慎重谋事。《断易天机》解：讼卦乾上坎下，刚遇险，必有争论，多不吉。北宋易学家邵雍解：天高水深，远离不亲，慎谋退守则无凶。得此卦者，身心不安，多争诉，宜修身养性。台湾国学大儒傅佩荣解：时运受阻，财运初谨慎终获利，家宅君子求淑女，身体预防胜于治疗。传统解卦：异卦（下坎上乾），刚健遇险，彼此反对，生争讼，需慎重戒惧。
